# Taller 2 — Modelo 2: Predicción de Partidos de la Premier League
## Regresión Lineal (total_goals) + Regresión Logística (H/D/A)
**Machine Learning I (ML1-2026I) — Universidad Externado de Colombia**

In [ ]:
# ============================================================
# CELDA 1: Importaciones y configuración general
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import json
import pickle
import requests
from pathlib import Path

from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold, cross_validate
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, f1_score, confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Directorios del proyecto
BASE_DIR = Path(__file__).resolve().parent.parent
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

API_BASE = "https://premier.72-60-245-2.sslip.io"

print("✅ Librerías importadas correctamente")
print(f"📁 Datos: {DATA_DIR}")
print(f"📁 Modelos: {MODELS_DIR}")

In [ ]:
# ============================================================
# CELDA 2: Carga de datos desde el API (con caché local)
# ============================================================
def cargar_csv(nombre: str, url: str) -> pd.DataFrame:
    """Carga un CSV desde archivo local si existe, si no lo descarga del API."""
    ruta = DATA_DIR / f"{nombre}.csv"
    if ruta.exists():
        print(f"📂 [{nombre}] cargando desde archivo local...")
        return pd.read_csv(ruta)
    print(f"⬇️  [{nombre}] descargando desde el API...")
    df = pd.read_csv(url)
    df.to_csv(ruta, index=False)
    print(f"💾 [{nombre}] guardado en {ruta}")
    return df

matches = cargar_csv("matches", f"{API_BASE}/export/matches")

print(f"\n✅ matches: {matches.shape[0]} filas × {matches.shape[1]} columnas")
print(f"Columnas: {list(matches.columns)}")

In [ ]:
# ============================================================
# CELDA 3: Limpieza y variables derivadas de matches
# ============================================================

# Fecha en formato correcto
matches['date'] = pd.to_datetime(matches['date'], format='%d/%m/%Y')
matches = matches.sort_values('date').reset_index(drop=True)

# Variable objetivo de regresión lineal
matches['total_goals'] = matches['fthg'] + matches['ftag']
matches['goal_diff']   = matches['fthg'] - matches['ftag']

# Probabilidades implícitas normalizadas a partir de odds Bet365
# (se elimina el margen de la casa para obtener probabilidades más puras)
for col_odd, col_prob in [('b365h', 'implied_prob_h'),
                           ('b365d', 'implied_prob_d'),
                           ('b365a', 'implied_prob_a')]:
    matches[col_prob] = 1 / matches[col_odd]

total_impl = matches[['implied_prob_h', 'implied_prob_d', 'implied_prob_a']].sum(axis=1)
matches['implied_prob_h'] /= total_impl
matches['implied_prob_d'] /= total_impl
matches['implied_prob_a'] /= total_impl

# Ventaja de jugar en casa (feature constante pero útil para el modelo lineal)
matches['home_advantage'] = 1

print("✅ Variables derivadas creadas")
print(f"\nEstadísticas de total_goals:")
print(matches['total_goals'].describe().round(2))
print(f"\nDistribución de resultados (ftr):")
print(matches['ftr'].value_counts(normalize=True).round(3))

In [ ]:
# ============================================================
# CELDA 4: Rolling averages sin data leakage
#
# Se calculan en orden cronológico usando SOLO partidos ANTERIORES
# a cada jornada (shift implícito — se actualiza el historial DESPUÉS
# de asignar las stats).
# ============================================================
def calcular_rolling_stats(df: pd.DataFrame, n: int = 5) -> pd.DataFrame:
    """
    Para cada partido, agrega las medias de los últimos n partidos
    de cada equipo ANTES de ese partido (sin data leakage).

    Columnas añadidas:
    - home_avg_goals_scored_last5
    - home_avg_goals_conceded_last5
    - away_avg_goals_scored_last5
    - away_avg_goals_conceded_last5
    - home_form_points_last5
    - away_form_points_last5
    """
    df = df.copy().sort_values('date').reset_index(drop=True)
    nuevas_cols = [
        'home_avg_goals_scored_last5', 'home_avg_goals_conceded_last5',
        'away_avg_goals_scored_last5', 'away_avg_goals_conceded_last5',
        'home_form_points_last5', 'away_form_points_last5',
    ]
    for col in nuevas_cols:
        df[col] = np.nan

    # historial[equipo] = lista de (goles_a_favor, goles_en_contra, puntos)
    historial: dict = {}

    for idx, row in df.iterrows():
        home = row['home_team']
        away = row['away_team']
        fthg = row['fthg']
        ftag = row['ftag']
        ftr  = row['ftr']

        pts_home = 3 if ftr == 'H' else (1 if ftr == 'D' else 0)
        pts_away = 3 if ftr == 'A' else (1 if ftr == 'D' else 0)

        # Asignar stats ANTES de este partido
        for equipo, rol in [(home, 'home'), (away, 'away')]:
            if equipo in historial and historial[equipo]:
                ult = historial[equipo][-n:]
                df.at[idx, f'{rol}_avg_goals_scored_last5']   = np.mean([g[0] for g in ult])
                df.at[idx, f'{rol}_avg_goals_conceded_last5'] = np.mean([g[1] for g in ult])
                df.at[idx, f'{rol}_form_points_last5']        = np.mean([g[2] for g in ult])

        # Actualizar historial CON este partido
        historial.setdefault(home, []).append((fthg, ftag, pts_home))
        historial.setdefault(away, []).append((ftag, fthg, pts_away))

    return df


matches = calcular_rolling_stats(matches, n=5)

# Eliminar filas sin historial (primeros partidos de cada equipo)
ROLLING_COLS = [
    'home_avg_goals_scored_last5', 'home_avg_goals_conceded_last5',
    'away_avg_goals_scored_last5', 'away_avg_goals_conceded_last5',
]
matches_modelo = matches.dropna(subset=ROLLING_COLS).copy()

print(f"✅ Rolling averages calculadas (ventana = 5 partidos)")
print(f"Partidos con historial completo: {len(matches_modelo)} / {len(matches)}")
print(matches_modelo[['home_team', 'away_team',
                       'home_avg_goals_scored_last5',
                       'away_avg_goals_scored_last5',
                       'total_goals', 'ftr']].head(5).to_string(index=False))

In [ ]:
# ============================================================
# CELDA 5: Definición de features para los modelos
# ============================================================
FEATURES_ODDS = [
    # Probabilidades implícitas Bet365 (normalizadas)
    'implied_prob_h', 'implied_prob_d', 'implied_prob_a',
    # Odds crudas de múltiples casas de apuestas
    'b365h', 'b365d', 'b365a',
    'avgh', 'avgd', 'avga',
    'maxh', 'maxd', 'maxa',
    # Efecto de jugar en casa
    'home_advantage',
]

FEATURES_ROLLING = [
    'home_avg_goals_scored_last5',
    'home_avg_goals_conceded_last5',
    'away_avg_goals_scored_last5',
    'away_avg_goals_conceded_last5',
    'home_form_points_last5',
    'away_form_points_last5',
]

FEATURES = FEATURES_ODDS + FEATURES_ROLLING
TARGET_LINEAL    = 'total_goals'
TARGET_LOGISTICO = 'ftr'

X          = matches_modelo[FEATURES].copy()
y_lineal   = matches_modelo[TARGET_LINEAL].copy()
y_logistico = matches_modelo[TARGET_LOGISTICO].copy()

print(f"✅ Features listas: {len(FEATURES)} variables")
print(f"   - Odds: {len(FEATURES_ODDS)}")
print(f"   - Rolling: {len(FEATURES_ROLLING)}")
print(f"Dataset: {X.shape[0]} partidos × {X.shape[1]} features")
print(f"\nDistribución del objetivo (ftr):")
print(y_logistico.value_counts(normalize=True).round(3))

In [ ]:
# ============================================================
# VISUALIZACIÓN 1: Distribución de goles totales por partido
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma con media y mediana
axes[0].hist(matches['total_goals'], bins=range(0, 12),
             color='#4C72B0', edgecolor='white', alpha=0.85, rwidth=0.85)
axes[0].axvline(matches['total_goals'].mean(), color='#DD4444', linewidth=2.5,
                linestyle='--', label=f"Media: {matches['total_goals'].mean():.2f}")
axes[0].axvline(matches['total_goals'].median(), color='#FFAA00', linewidth=2.5,
                linestyle=':', label=f"Mediana: {matches['total_goals'].median():.0f}")
axes[0].set_xlabel("Goles totales por partido")
axes[0].set_ylabel("Frecuencia")
axes[0].set_title("Distribución de Goles por Partido\nPremier League 2025-26 (GW1-30)")
axes[0].legend()
axes[0].set_xticks(range(0, 12))

# Porcentaje por número de goles
freq = matches['total_goals'].value_counts().sort_index()
axes[1].bar(freq.index, freq.values / len(matches) * 100,
            color='#4C72B0', edgecolor='white', alpha=0.85)
axes[1].set_xlabel("Goles totales")
axes[1].set_ylabel("% de partidos")
axes[1].set_title("Porcentaje de Partidos por Número de Goles")
axes[1].set_xticks(range(0, 12))
for idx, val in zip(freq.index, freq.values):
    pct = val / len(matches) * 100
    axes[1].text(idx, pct + 0.3, f'{pct:.1f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(DATA_DIR / 'eda_1_distribucion_goles.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Visualización 1 guardada")

In [ ]:
# ============================================================
# VISUALIZACIÓN 2: Distribución de resultados H/D/A
# ============================================================
COLORES = {'H': '#2196F3', 'D': '#9E9E9E', 'A': '#F44336'}
ETIQUETAS = {'H': 'Local gana (H)', 'D': 'Empate (D)', 'A': 'Visitante (A)'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
conteo = matches['ftr'].value_counts()

# Barras
barras = axes[0].bar(
    [ETIQUETAS[k] for k in ['H', 'D', 'A']],
    [conteo.get(k, 0) for k in ['H', 'D', 'A']],
    color=[COLORES[k] for k in ['H', 'D', 'A']],
    edgecolor='white', alpha=0.9
)
for barra, k in zip(barras, ['H', 'D', 'A']):
    pct = conteo.get(k, 0) / len(matches) * 100
    axes[0].text(barra.get_x() + barra.get_width() / 2,
                 barra.get_height() + 1,
                 f'{pct:.1f}%\n({conteo.get(k, 0)})',
                 ha='center', fontsize=11, fontweight='bold')
axes[0].set_ylabel("Número de partidos")
axes[0].set_title("Distribución de Resultados\nPremier League 2025-26")
axes[0].set_ylim(0, conteo.max() * 1.2)

# Pastel
wedges, texts, autotexts = axes[1].pie(
    [conteo.get(k, 0) for k in ['H', 'D', 'A']],
    labels=[ETIQUETAS[k] for k in ['H', 'D', 'A']],
    colors=[COLORES[k] for k in ['H', 'D', 'A']],
    autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for at in autotexts:
    at.set_fontsize(12)
    at.set_fontweight('bold')
axes[1].set_title("Proporción de Resultados")

plt.tight_layout()
plt.savefig(DATA_DIR / 'eda_2_distribucion_resultados.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Visualización 2 guardada")

In [ ]:
# ============================================================
# VISUALIZACIÓN 3: Probabilidades implícitas Bet365 vs resultado real
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

prob_info = [
    ('implied_prob_h', 'Prob. Local (H)',      '#2196F3'),
    ('implied_prob_d', 'Prob. Empate (D)',      '#9E9E9E'),
    ('implied_prob_a', 'Prob. Visitante (A)',   '#F44336'),
]
etiq_eje = ['Local\ngana', 'Empate', 'Visitante\ngana']

for ax, (col, titulo, _) in zip(axes, prob_info):
    datos = [matches[matches['ftr'] == r][col].values for r in ['H', 'D', 'A']]
    bp = ax.boxplot(datos, patch_artist=True, labels=etiq_eje,
                    medianprops=dict(color='black', linewidth=2))
    colores_box = [COLORES['H'], COLORES['D'], COLORES['A']]
    for patch, c in zip(bp['boxes'], colores_box):
        patch.set_facecolor(c)
        patch.set_alpha(0.7)
    ax.set_title(f'{titulo}\nvs resultado real')
    ax.set_ylabel('Probabilidad implícita')
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.4)

plt.suptitle('Probabilidades Implícitas Bet365 vs Resultado Real',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(DATA_DIR / 'eda_3_probabilidades_vs_resultado.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Visualización 3 guardada")

In [ ]:
# ============================================================
# VISUALIZACIÓN 4: Heatmap de correlación de features numéricas
# ============================================================
cols_corr = FEATURES + ['total_goals']
corr = matches_modelo[cols_corr].corr()

fig, ax = plt.subplots(figsize=(16, 13))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, ax=ax, annot_kws={'size': 8},
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Mapa de Correlación — Features de Partidos\n(triángulo inferior)',
             fontsize=14, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(DATA_DIR / 'eda_4_heatmap_correlacion.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Visualización 4 guardada")

In [ ]:
# ============================================================
# VISUALIZACIÓN 5: Rolling de goles — Top 5 equipos ofensivos
# ============================================================
def rolling_goles_equipo(df: pd.DataFrame, equipo: str, n: int = 5) -> pd.Series:
    """Media móvil de goles anotados (últimos n partidos) para un equipo."""
    partidos = []
    for _, row in df.sort_values('date').iterrows():
        if row['home_team'] == equipo:
            partidos.append({'fecha': row['date'], 'goles': row['fthg']})
        elif row['away_team'] == equipo:
            partidos.append({'fecha': row['date'], 'goles': row['ftag']})
    serie = pd.DataFrame(partidos).set_index('fecha')
    return serie['goles'].rolling(n, min_periods=1).mean()

# Calcular total de goles por equipo
goles_por_equipo = {
    eq: (matches[matches['home_team'] == eq]['fthg'].sum() +
         matches[matches['away_team'] == eq]['ftag'].sum())
    for eq in matches['home_team'].unique()
}
top5_equipos = sorted(goles_por_equipo, key=goles_por_equipo.get, reverse=True)[:5]

PALETA = ['#EF4444', '#3B82F6', '#10B981', '#F59E0B', '#8B5CF6']

fig, ax = plt.subplots(figsize=(14, 6))
for equipo, color in zip(top5_equipos, PALETA):
    serie = rolling_goles_equipo(matches, equipo, n=5)
    ax.plot(serie.index, serie.values,
            marker='o', markersize=3, linewidth=2,
            label=f'{equipo} ({goles_por_equipo[equipo]} goles)',
            color=color)

ax.set_xlabel("Fecha")
ax.set_ylabel("Media móvil de goles (últimos 5 partidos)")
ax.set_title("Rendimiento Ofensivo — Top 5 Equipos\nMedia Móvil (ventana = 5)",
             fontsize=13, fontweight='bold')
ax.legend(loc='upper left')
ax.grid(alpha=0.4)
plt.tight_layout()
plt.savefig(DATA_DIR / 'eda_5_rolling_top5.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Visualización 5 guardada")

In [ ]:
# ============================================================
# MODELO 2A — Regresión Lineal (Ridge): Predecir total_goals
# ============================================================
# Escalar features
scaler_lineal = StandardScaler()
X_sc_lin = scaler_lineal.fit_transform(X)

# Baseline naive: predecir siempre la media
rmse_baseline = float(np.sqrt(((y_lineal - y_lineal.mean()) ** 2).mean()))

# Modelo Ridge (regularización L2 — robusto con pocas muestras y features correlacionadas)
model_linear = Ridge(alpha=1.0, fit_intercept=True)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_lin = cross_validate(
    model_linear, X_sc_lin, y_lineal, cv=kf,
    scoring={
        'neg_rmse': 'neg_root_mean_squared_error',
        'neg_mae':  'neg_mean_absolute_error',
        'r2':       'r2',
    },
    return_train_score=True
)

rmse_cv = float(-cv_lin['test_neg_rmse'].mean())
mae_cv  = float(-cv_lin['test_neg_mae'].mean())
r2_cv   = float(cv_lin['test_r2'].mean())

print("=" * 58)
print("  MODELO 2A — RIDGE REGRESSION (alpha=1.0)")
print("  Objetivo: Predecir goles totales por partido")
print("=" * 58)
print(f"\n  Cross-Validation k=5:")
print(f"    RMSE : {rmse_cv:.4f} ± {cv_lin['test_neg_rmse'].std():.4f}")
print(f"    MAE  : {mae_cv:.4f} ± {cv_lin['test_neg_mae'].std():.4f}")
print(f"    R²   : {r2_cv:.4f} ± {cv_lin['test_r2'].std():.4f}")
print(f"\n  Baseline naive (siempre media={y_lineal.mean():.2f}):")
print(f"    RMSE baseline: {rmse_baseline:.4f}")
print(f"    ¿Mejora baseline? {'✅ SÍ' if rmse_cv < rmse_baseline else '❌ NO'}")

# Entrenar sobre todos los datos para generar predicciones finales
model_linear.fit(X_sc_lin, y_lineal)

# Top features por coeficiente
coef_df = pd.DataFrame({
    'Feature':     FEATURES,
    'Coeficiente': model_linear.coef_
}).sort_values('Coeficiente', key=abs, ascending=False)

print(f"\n  Top 10 features (|coeficiente| mayor):")
print(coef_df.head(10).to_string(index=False))

In [ ]:
# ============================================================
# MODELO 2B — Regresión Logística Multiclase: Predecir ftr (H/D/A)
# ============================================================
BET365_ACCURACY = 0.498

scaler_logistico = StandardScaler()
X_sc_log = scaler_logistico.fit_transform(X)

# class_weight='balanced' compensa el desbalance H > A > D
# solver='lbfgs' es el predeterminado para multiclase en sklearn >= 1.2
model_logistic = LogisticRegression(
    penalty='l2',
    C=1.0,
    class_weight='balanced',
    solver='lbfgs',
    max_iter=1000,
    random_state=42
)

kf_log = KFold(n_splits=5, shuffle=True, random_state=42)
cv_log = cross_validate(
    model_logistic, X_sc_log, y_logistico, cv=kf_log,
    scoring=['accuracy', 'f1_macro', 'f1_weighted'],
    return_train_score=True
)

acc_cv    = float(cv_log['test_accuracy'].mean())
f1mac_cv  = float(cv_log['test_f1_macro'].mean())
f1wgt_cv  = float(cv_log['test_f1_weighted'].mean())
diferencia = acc_cv - BET365_ACCURACY

print("=" * 58)
print("  MODELO 2B — REGRESIÓN LOGÍSTICA MULTICLASE")
print("  Objetivo: Predecir resultado (H / D / A)")
print("=" * 58)
print(f"\n  Cross-Validation k=5:")
print(f"    Accuracy :   {acc_cv:.4f} ± {cv_log['test_accuracy'].std():.4f}")
print(f"    F1 Macro :   {f1mac_cv:.4f} ± {cv_log['test_f1_macro'].std():.4f}")
print(f"    F1 Weighted: {f1wgt_cv:.4f} ± {cv_log['test_f1_weighted'].std():.4f}")
print(f"\n{'─'*58}")
print(f"  COMPARACIÓN VS BET365:")
print(f"    Bet365 accuracy  : {BET365_ACCURACY:.1%}")
print(f"    Nuestro modelo   : {acc_cv:.1%}")
print(f"    Diferencia       : {diferencia:+.1%}")
print(f"    ¿Supera Bet365?  : {'✅ SÍ' if diferencia > 0 else '⚠️  No (aún cerca)'}")
print("=" * 58)

# Entrenar sobre todos los datos
model_logistic.fit(X_sc_log, y_logistico)
y_pred_log = model_logistic.predict(X_sc_log)

# Reporte detallado
print(f"\n  Reporte de Clasificación (entrenamiento completo):")
print(classification_report(y_logistico, y_pred_log,
                            target_names=['Away (A)', 'Draw (D)', 'Home (H)']))

# Matriz de confusión
cm = confusion_matrix(y_logistico, y_pred_log, labels=['H', 'D', 'A'])
print("  Matriz de Confusión  [filas=real | columnas=predicho]:")
print(f"  {'':>4} {'H':>6} {'D':>6} {'A':>6}")
for i, etiq in enumerate(['H', 'D', 'A']):
    print(f"  {etiq:>4} {cm[i][0]:>6} {cm[i][1]:>6} {cm[i][2]:>6}")

print("\n  ℹ️  Nota: Los empates (D) son los más difíciles de predecir.")
print("  Analiza el recall de D en el reporte — suele ser el más bajo.")

In [ ]:
# ============================================================
# CELDA FINAL: Generar match_predictions y guardar modelos
# ============================================================

# --- Predicciones lineales ---
pred_goals = np.clip(model_linear.predict(X_sc_lin), 0, None)

# --- Predicciones logísticas ---
clases     = model_logistic.classes_           # e.g. ['A', 'D', 'H']
pred_res   = model_logistic.predict(X_sc_log)
prob_mat   = model_logistic.predict_proba(X_sc_log)

idx_H = list(clases).index('H')
idx_D = list(clases).index('D')
idx_A = list(clases).index('A')

match_predictions = pd.DataFrame({
    'match_id':        matches_modelo['id'].values,
    'home_team':       matches_modelo['home_team'].values,
    'away_team':       matches_modelo['away_team'].values,
    'date':            matches_modelo['date'].dt.strftime('%d/%m/%Y').values,
    'pred_total_goals': np.round(pred_goals, 2),
    'real_total_goals': matches_modelo['total_goals'].values,
    'real_home_goals': matches_modelo['fthg'].values,
    'real_away_goals': matches_modelo['ftag'].values,
    'pred_result':     pred_res,
    'real_result':     matches_modelo['ftr'].values,
    'prob_H':          np.round(prob_mat[:, idx_H], 4),
    'prob_D':          np.round(prob_mat[:, idx_D], 4),
    'prob_A':          np.round(prob_mat[:, idx_A], 4),
})

# Guardar predicciones como CSV y JSON
match_predictions.to_csv(DATA_DIR / 'match_predictions.csv', index=False)
match_predictions.to_json(DATA_DIR / 'match_predictions.json',
                          orient='records', indent=2)

# Guardar modelos y scalers
artefactos = {
    'model_linear.pkl':    model_linear,
    'model_logistic.pkl':  model_logistic,
    'scaler_lineal.pkl':   scaler_lineal,
    'scaler_logistico.pkl': scaler_logistico,
    'features_modelo2.pkl': {
        'FEATURES':       FEATURES,
        'FEATURES_ODDS':  FEATURES_ODDS,
        'FEATURES_ROLLING': FEATURES_ROLLING,
        'clases_logistico': list(clases),
    },
}
for nombre, objeto in artefactos.items():
    with open(MODELS_DIR / nombre, 'wb') as f:
        pickle.dump(objeto, f)

# ============================================================
# RESUMEN FINAL
# ============================================================
print("\n" + "=" * 58)
print("        RESUMEN FINAL — TALLER 2 MODELO 2")
print("=" * 58)
print(f"\n📊 Dataset: {len(matches_modelo)} partidos | {len(FEATURES)} features")
print(f"   → {len(FEATURES_ODDS)} features de odds")
print(f"   → {len(FEATURES_ROLLING)} rolling averages (sin leakage)")

print(f"\n🔵 MODELO 2A — Ridge Regression (total_goals)")
print(f"   RMSE  (CV k=5): {rmse_cv:.4f}  |  baseline: {rmse_baseline:.4f}")
print(f"   MAE   (CV k=5): {mae_cv:.4f}")
print(f"   R²    (CV k=5): {r2_cv:.4f}")
print(f"   Mejora baseline: {'✅ SÍ' if rmse_cv < rmse_baseline else '❌ NO'}")

print(f"\n🔴 MODELO 2B — Logistic Regression (H/D/A)")
print(f"   Accuracy (CV k=5): {acc_cv:.1%}")
print(f"   F1 Macro (CV k=5): {f1mac_cv:.1%}")
print(f"   Bet365 accuracy:   {BET365_ACCURACY:.1%}")
print(f"   Diferencia:        {diferencia:+.1%}  {'✅' if diferencia > 0 else '⚠️'}")

print(f"\n💾 Archivos generados:")
print(f"   data/match_predictions.csv  ({len(match_predictions)} filas)")
print(f"   data/match_predictions.json")
for nombre in artefactos:
    print(f"   models/{nombre}")
print(f"\n{'='*58}")
print("  Visualizaciones EDA guardadas en data/eda_*.png")
print("=" * 58)